In [50]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.metrics import precision_recall_fscore_support
import joblib
import os

In [51]:
df = pd.read_csv('../data/processed/telco_features.csv')

X = df[[ c for c in df.columns if c != 'Churn']]
y = df['Churn']


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print('--------------------')
print(f'y_train mean: {y_train.mean()}')
print(f'y_test mean: {y_test.mean()}')

X_train shape: (5625, 29)
X_test shape: (1407, 29)
--------------------
y_train mean: 0.2657777777777778
y_test mean: 0.2658137882018479


In [52]:
# Scaler fitted on X_train only — transform applied to both train and test
# to prevent test-set distribution from leaking into the scaler parameters
pipe = Pipeline([
    ('scaler',  StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])


pipe.fit(X_train, y_train)

y_pred_proba_v1 = pipe.predict_proba(X_test)[:, 1]

score_v1 = roc_auc_score(y_test, y_pred_proba_v1)

print('Baseline (unscaled): 0.8348')
print(f'V1 (scaled): {score_v1}')

Baseline (unscaled): 0.8348
V1 (scaled): 0.8340472948838075


In [53]:

# Tree-based models are scale-invariant — no StandardScaler needed here
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)

rf.fit(X_train, y_train)

y_pred_proba_v2 = rf.predict_proba(X_test)[:, 1]

score_v2 = roc_auc_score(y_test, y_pred_proba_v2)

print(f'V1 (scaled LogReg): {score_v1}')
print(f'V2 (Random Forest): {score_v2}')

V1 (scaled LogReg): 0.8340472948838075
V2 (Random Forest): 0.8217317816856567


In [54]:
# Default params intentionally used here — tuning is handled in notebook 07
lgbm = LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1)
lgbm.fit(X_train, y_train)

y_pred_proba_v3 = lgbm.predict_proba(X_test)[:, 1]

score_v3 = roc_auc_score(y_test, y_pred_proba_v3)

print(f'V2 (Random Forest): {score_v2}')
print(f'V3 (LightGBM): {score_v3}')

V2 (Random Forest): 0.8217317816856567
V3 (LightGBM): 0.8344316693499542


In [55]:
results = pd.DataFrame({
    'Model': [
        'Baseline Logistic Regression',
        'V1 Scaled Logistic Regression',
        'V2 Random Forest',
        'V3 LightGBM'
    ],
    'ROC_AUC': [
        0.8348,
        score_v1,
        score_v2,
        score_v3
    ]
})

results = results.sort_values('ROC_AUC', ascending=False)

print(results)

best_model = results.iloc[0]

print(
    f"\nBest performer: {best_model['Model']} "
    f"(ROC-AUC = {best_model['ROC_AUC']:.4f})"
)

                           Model   ROC_AUC
0   Baseline Logistic Regression  0.834800
3                    V3 LightGBM  0.834432
1  V1 Scaled Logistic Regression  0.834047
2               V2 Random Forest  0.821732

Best performer: Baseline Logistic Regression (ROC-AUC = 0.8348)


In [56]:
importance = pd.Series(
    lgbm.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)


print("Top 10 features by LightGBM importance:")
print(importance.head(10))

Top 10 features by LightGBM importance:
MonthlyCharges                    575
TotalCharges                      555
AvgMonthlySpend                   535
tenure                            400
gender                             90
PaperlessBilling                   71
OnlineSecurity                     61
PaymentMethod_Electronic check     60
MultipleLines                      54
Partner                            52
dtype: int32


- LightGBM uses split-count importance by default — this systematically undervalues binary features (one split point) vs continuous ones (many split points).
- Contract_Month-to-month not appearing in top 10 despite being the strongest predictor in EDA is a direct consequence of this metric, not a model flaw.

In [57]:
# Using scaled pipeline probabilities — model converged cleanly,
# probability estimates are more reliable than the unscaled lr_base
# which triggered a ConvergenceWarning

print('Threshold tuning on Baseline Logistic Regression\n')
print(f"{'Threshold':<12} {'Precision':<12} {'Recall':<12} {'F1':<10}")
print('-' * 46)

for threshold in [0.3, 0.35, 0.4, 0.45, 0.5]:
    y_pred_thresh = (y_pred_proba_v1 >= threshold).astype(int)
    p, r, f, _ = precision_recall_fscore_support(y_test, y_pred_thresh, pos_label=1, average='binary')
    print(f"{threshold:<12} {p:<12.3f} {r:<12.3f} {f:<10.3f}")

Threshold tuning on Baseline Logistic Regression

Threshold    Precision    Recall       F1        
----------------------------------------------
0.3          0.503        0.759        0.605     
0.35         0.546        0.717        0.620     
0.4          0.573        0.668        0.617     
0.45         0.606        0.610        0.608     
0.5          0.628        0.532        0.576     


- Threshold of 0.35 recommended: best F1 (0.620) and recall jumps from 53.2% to 71.7% — catching ~70 more churners at the cost of lower precision.
- In churn context, missing a churner costs more than a false alarm.

In [58]:
# Saving the pipeline (scaler + model) as a single object —
# both components are needed at inference time

os.makedirs('../models', exist_ok=True)
joblib.dump(pipe, '../models/best_model.joblib')

print('Model: Scaled Logistic Regression (Pipeline)')
print(f'ROC-AUC: {score_v1:.4f}')
print('Recommended threshold: 0.35 (Recall: 0.717, Precision: 0.546, F1: 0.620)')
print(f'Saved to ../models/best_model.joblib')
print(f'File size: {os.path.getsize("../models/best_model.joblib"):,} bytes')

Model: Scaled Logistic Regression (Pipeline)
ROC-AUC: 0.8340
Recommended threshold: 0.35 (Recall: 0.717, Precision: 0.546, F1: 0.620)
Saved to ../models/best_model.joblib
File size: 3,073 bytes


## Model Improvement Summary

**Split:** 80/20 stratified, random_state=42 — identical to notebook 05 for valid comparison

| Version | Model | ROC-AUC | Notes |
|---------|-------|---------|-------|
| Baseline | Logistic Regression (unscaled) | 0.8348 | ConvergenceWarning — scale mismatch |
| V1 | Logistic Regression + StandardScaler (Pipeline) | 0.8340 | Converged cleanly — scale fixed |
| V2 | Random Forest (200 trees) | 0.8217 | No scaling needed — tree-based |
| V3 | LightGBM (default params) | 0.8344 | Default params only — tuning in notebook 07 |

**Key finding:** All three non-RF models score within 0.001 ROC-AUC of each other.
The decision boundary for this dataset is largely linear — the strongest predictors
(contract type, tenure, payment method) are clean encoded signals without complex
interactions that tree models can exploit with default settings.

**class_weight='balanced' tested and rejected** on both RF and LightGBM: added weight
hurt ROC-AUC on both. At 73/27 imbalance, the models were already learning the churn
signal adequately — aggressive reweighting over-corrected.

**LightGBM feature importance caveat:** Split-count importance undervalues binary features.
Contract_Month-to-month (strongest signal in EDA at 0.40 correlation) does not appear
in the top 10 — MonthlyCharges, TotalCharges, and AvgMonthlySpend dominate because
continuous features offer many more split points. This is a known limitation of the metric,
not a finding about the actual predictive importance of contract type.

**Threshold tuning (Scaled Pipeline):**

| Threshold | Precision | Recall | F1 |
|-----------|-----------|--------|----|
| 0.3 | 0.503 | 0.759 | 0.605     
| 0.35 | 0.546 | 0.717 | 0.620     
| 0.4 | 0.573 | 0.668 | 0.617     
| 0.45 | 0.606 | 0.610 | 0.608     
| 0.5 | 0.628 | 0.532 | 0.576 


**Recommended threshold: 0.35** — best F1 and raises recall from ~53% to ~72%,
catching significantly more churners. In a churn context, the cost of missing a
churner exceeds the cost of a false alarm.

**Best model saved:** `models/best_model.joblib` — Pipeline (StandardScaler + Logistic Regression)